# Compile ECM data by section

Use the same cleaned-file preference, ECM loading, 30 mm end trimming, and outside-track normalization as `make_oneline.ipynb`. Exclude surface imperfections and non-finite measurements, then calculate one arithmetic mean per channel and section over all remaining samples (all faces and tracks). There is no depth binning or smoothing; faces/tracks contribute in proportion to their usable sample counts.

Keep every section listed for the selected core in metadata. Missing channels, missing files, or channels with no usable samples give `NaN`; partial gaps are omitted from the mean without interpolation. CSV files contain literal `NaN` and no index.


## Import packages


In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd

sys.path.append("../core_scripts/")
from ECMclass import ECM


## User inputs
Run this notebook from `scripts/processing`, as with `make_oneline.ipynb`.


In [ ]:
path_to_data = '../../data/'
path_to_metadata = '../../data/'
metadata_file = 'metadata.csv'
trim_mm = 30

meta = pd.read_csv(path_to_metadata + metadata_file)

# Compile all cores in metadata, or replace with a list such as ['ALHIC2501'].
cores = meta['core'].drop_duplicates().tolist()


## Compile and save
Output: `data/processed/<core>_by_section.csv`. Section order follows metadata.


In [ ]:
def make_by_section(core):
    core_meta = meta.loc[meta['core'] == core]
    if core_meta.empty:
        raise ValueError(f"No metadata found for core: {core}")

    rows = []
    for section, section_meta in core_meta.groupby('section', sort=False):
        totals = {'AC': 0.0, 'DC': 0.0}
        counts = {'AC': 0, 'DC': 0}

        for _, row in section_meta.iterrows():
            channel = row['ACorDC']
            if channel not in totals:
                raise ValueError(f"Unexpected ECM channel: {channel}")

            filename = f"{core}-{section}-{row['face']}-{channel}.csv"
            if os.path.exists(path_to_data + 'clean/' + core + '/' + filename):
                path_to_file = path_to_data + 'clean/'
            else:
                path_to_file = path_to_data + 'not-cleaned/'

            if not os.path.exists(path_to_file + core + '/' + filename):
                warnings.warn(f"Missing ECM file: {filename}; using available samples only.")
                continue

            data_item = ECM(core, section, row['face'], channel,
                            path_to_file, path_to_metadata + metadata_file)
            data_item.rem_ends(trim_mm)
            if data_item.meas.size == 0:
                continue
            data_item.norm_outside()

            measurements = np.asarray(data_item.meas, dtype=float)
            keep = (np.asarray(data_item.surface_imperfection) == 0) & np.isfinite(measurements)
            totals[channel] += measurements[keep].sum()
            counts[channel] += int(keep.sum())

        rows.append({
            'core': core,
            'section': section,
            'AC': totals['AC'] / counts['AC'] if counts['AC'] else np.nan,
            'DC': totals['DC'] / counts['DC'] if counts['DC'] else np.nan,
        })

    df = pd.DataFrame(rows, columns=['core', 'section', 'AC', 'DC'])
    os.makedirs(path_to_data + 'processed/', exist_ok=True)
    output_file = path_to_data + 'processed/' + core + '_by_section.csv'
    df.to_csv(output_file, index=False, na_rep='NaN')
    print(f"Saved {len(df)} sections to {output_file}")
    return df


In [ ]:
by_section = {core: make_by_section(core) for core in cores}
